# Crib sheet: Feldman-Cousins confidence intervals

A quick reference for computing Feldman-Cousins (FC) confidence intervals
for a Poisson signal in the presence of known background.

Reference: Feldman & Cousins, *Phys. Rev. D* **57**, 3873 (1998).

## Setup

In [ ]:
%matplotlib inline

import numpy  as np
import matplotlib.pyplot as plt
import scipy.stats as stats

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)

import core.confint as confint
import core.pltext  as pltext
pltext.style()

print('Fanal root:', rootpath)

---
## 1. The problem

We observe $n$ events in a counting experiment.  
The expected count is $\mu + b$, where $\mu \geq 0$ is the signal and $b$ is the known background.

$$n \sim \mathrm{Poisson}(\mu + b)$$

Given $n_\mathrm{obs}$, the FC method provides a confidence interval $[\mu_\mathrm{low},\, \mu_\mathrm{high}]$ that:
- has correct **coverage** at all $\mu$ values (even near $\mu = 0$),
- automatically transitions from a two-sided interval to an upper limit.

---
## 2. The FC ordering principle

For each hypothesised $\mu$, rank observations $n$ by the **likelihood ratio**:

$$R(n) = \frac{P(n \mid \mu + b)}{P(n \mid \hat\mu_{\mathrm{best}} + b)}$$

where $\hat\mu_{\mathrm{best}} = \max(n - b,\, 0)$.

Include observations in order of decreasing $R$ until the cumulative probability reaches the desired confidence level $\alpha$.

The set of included $n$ values defines the **acceptance interval** $[n_1, n_2]$ for that $\mu$.

---
## 3. `confint` module API

| Function | Purpose | Returns |
|----------|---------|--------|
| `fc_confsegment(mu, bkg, cl)` | Acceptance interval for one $\mu$ | `(n_min, n_max)` |
| `fc_confband(mus, bkg, cl)` | Acceptance band over a grid of $\mu$ | `(n0s, n1s)` arrays |
| `get_fc_confinterval(mus, bkg, cl)` | **Main entry point**: builds a closure | `ci(n_obs)` callable |
| `fca_segment(tmus, ns, cl)` | FC segment from MC ordering ($t_\mu$) | `[n_min, n_max]` |

---
## 4. Example: zero background

In [ ]:
# Define the grid of true signal values and the background
bkg = 0.
mus = np.linspace(0., 20., 200)

# Build FC confidence interval functions
fc_68 = confint.get_fc_confinterval(mus, bkg, cl=0.68)
fc_90 = confint.get_fc_confinterval(mus, bkg, cl=0.90)
fc_95 = confint.get_fc_confinterval(mus, bkg, cl=0.95)

In [ ]:
# Query for a specific observation
n_obs = 3

ci_68 = fc_68(n_obs)
ci_90 = fc_90(n_obs)
ci_95 = fc_95(n_obs)

print('Observation: n = {:d},  background b = {:.0f}'.format(n_obs, bkg))
print('  FC CI at 68% CL: ({:.2f}, {:.2f})'.format(*ci_68))
print('  FC CI at 90% CL: ({:.2f}, {:.2f})'.format(*ci_90))
print('  FC CI at 95% CL: ({:.2f}, {:.2f})'.format(*ci_95))

### Plotting the confidence belt

The FC confidence belt maps each observed count $n$ to the confidence interval on $\mu$.

In [ ]:
ns = np.arange(0, 20)

plt.figure(figsize=(7, 5))
for fc, cl, alpha in [(fc_95, 0.95, 0.2), (fc_90, 0.90, 0.35), (fc_68, 0.68, 0.5)]:
    ys = fc(ns)
    plt.fill_between(ns, ys[0], ys[1], alpha=alpha, color='steelblue',
                     label='FC CI {:.0f}% CL'.format(100 * cl))

plt.axvline(n_obs, color='red', ls='--', lw=1.5, label='$n_\mathrm{obs}$ = %d' % n_obs)
plt.xlabel('$n_\mathrm{obs}$')
plt.ylabel(r'$\mu$ (signal)')
plt.title('FC confidence belt  ($b = {:.0f}$)'.format(bkg))
plt.legend()
plt.tight_layout();

---
## 5. Example: with background

When $b > 0$, the FC interval naturally handles the physical boundary $\mu \geq 0$.  
For small $n_\mathrm{obs}$, the interval becomes a one-sided **upper limit**.

In [ ]:
bkg_val = 3.0
mus_b   = np.linspace(0., 25., 300)

fc_90_b = confint.get_fc_confinterval(mus_b, bkg_val, cl=0.90)

print('Background b = {:.1f}'.format(bkg_val))
print()
for n in range(0, 12):
    ci = fc_90_b(n)
    if ci[0] < 1e-6:
        print('  n = {:2d}:  mu < {:.2f}  (90% CL upper limit)'.format(n, ci[1]))
    else:
        print('  n = {:2d}:  mu in ({:.2f}, {:.2f})  (90% CL)'.format(n, ci[0], ci[1]))

In [ ]:
ns_b = np.arange(0, 20)

plt.figure(figsize=(7, 5))
ys = fc_90_b(ns_b)
plt.fill_between(ns_b, ys[0], ys[1], alpha=0.4, color='steelblue',
                 label='FC CI 90% CL')
plt.xlabel('$n_\mathrm{obs}$')
plt.ylabel(r'$\mu$ (signal)')
plt.title('FC confidence belt  ($b = {:.1f}$)'.format(bkg_val))
plt.legend()
plt.tight_layout();

---
## 6. The acceptance band

The **acceptance band** is the dual view: for each $\mu$, it shows which observations $n$ are accepted.

`fc_confband()` returns the lower and upper edges of this band.

In [ ]:
mus_band = np.linspace(0., 15., 150)
n0s, n1s = confint.fc_confband(mus_band, bkg=0., cl=0.90)

plt.figure(figsize=(7, 5))
plt.fill_between(mus_band, n0s, n1s, alpha=0.4, color='goldenrod',
                 label='Acceptance band 90% CL')
plt.xlabel(r'$\mu$ (true signal)')
plt.ylabel('$n$ (observed)')
plt.title('FC acceptance band  ($b = 0$)')
plt.legend()
plt.tight_layout();

---
## 7. Single-point acceptance interval

`fc_confsegment()` returns the acceptance interval for a single $\mu$ value.

In [ ]:
mu_test = 5.0
bkg_test = 2.0

seg_68 = confint.fc_confsegment(mu_test, bkg_test, cl=0.68)
seg_90 = confint.fc_confsegment(mu_test, bkg_test, cl=0.90)
seg_95 = confint.fc_confsegment(mu_test, bkg_test, cl=0.95)

print('mu = {:.1f}, b = {:.1f}'.format(mu_test, bkg_test))
print('  68% acceptance interval: n in [{:d}, {:d}]'.format(*seg_68))
print('  90% acceptance interval: n in [{:d}, {:d}]'.format(*seg_90))
print('  95% acceptance interval: n in [{:d}, {:d}]'.format(*seg_95))

---
## 8. Comparison with classical intervals

The FC interval differs from naive Poisson intervals near the physical boundary.
Let us compare with the standard $\sqrt{n}$ symmetric interval.

In [ ]:
bkg_cmp = 3.0
mus_cmp = np.linspace(0., 20., 200)
fc_90_cmp = confint.get_fc_confinterval(mus_cmp, bkg_cmp, cl=0.90)

ns_cmp = np.arange(0, 16)

plt.figure(figsize=(8, 5))

# FC intervals
ys_fc = fc_90_cmp(ns_cmp)
plt.fill_between(ns_cmp, ys_fc[0], ys_fc[1], alpha=0.3, color='steelblue',
                 label='FC 90% CL')
plt.plot(ns_cmp, ys_fc[0], 's-', ms=4, color='steelblue')
plt.plot(ns_cmp, ys_fc[1], 's-', ms=4, color='steelblue')

# Naive symmetric: mu_hat = n - b, +/- 1.645 * sqrt(n)
mu_hat = np.maximum(ns_cmp - bkg_cmp, 0.)
naive_lo = mu_hat - 1.645 * np.sqrt(ns_cmp)
naive_hi = mu_hat + 1.645 * np.sqrt(ns_cmp)
plt.plot(ns_cmp, naive_lo, 'v--', ms=4, color='tomato', label=r'Naive $\hat\mu \pm 1.645\sqrt{n}$')
plt.plot(ns_cmp, naive_hi, '^--', ms=4, color='tomato')

plt.axhline(0, color='gray', lw=0.5)
plt.xlabel('$n_\mathrm{obs}$')
plt.ylabel(r'$\mu$ (signal)')
plt.title('FC vs naive intervals  ($b = {:.1f}$)'.format(bkg_cmp))
plt.legend()
plt.tight_layout();

Notice how the naive interval gives **unphysical negative** values of $\mu$ for small $n_\mathrm{obs}$,
while the FC interval respects $\mu \geq 0$ and transitions smoothly to an upper limit.

---
## Summary

```
1. Build the FC confidence interval function
   fc = confint.get_fc_confinterval(mus, bkg, cl=0.90)

2. Query for an observation
   ci = fc(n_obs)           # returns [mu_low, mu_high]
   ci = fc(np.arange(20))   # accepts arrays too

3. Acceptance band (dual view)
   n0s, n1s = confint.fc_confband(mus, bkg, cl=0.90)

4. Single acceptance interval
   n_lo, n_hi = confint.fc_confsegment(mu, bkg, cl=0.90)
```

Key points:
- FC intervals have correct **coverage** at all $\mu$, including $\mu = 0$.
- They automatically become **upper limits** when $n_\mathrm{obs}$ is small.
- The `mus` grid must be fine enough to give smooth intervals.
- Background `bkg` must be known (not fitted).